# Metajeux-04 : Le chemin minimal, témoin construit et vérifié indépendamment — de Robinson-Goforth au vérificateur séparé

**Navigation** : [<< Metajeux-03-Abstraction-a-Dette](Metajeux-03-Abstraction-a-Dette-Python.ipynb) | [Index](README.md) | [jumeau Python](Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) | [Metajeux-05-Temoins-Impossibilite >>](Metajeux-05-Temoins-Impossibilite-Python.ipynb)

**La loi que ce notebook met à l'épreuve** — Loi II du deuxième voyage de digestion ICT :

> *recoordonner + passer du vérificateur au constructeur* : le précédent important n'est pas seulement
> le changement de coordonnées ; c'est changement de coordonnées **plus** passage du vérificateur au constructeur.

Elle est franchie une fois sur le substrat Life (GOL-01 : la machine **redécouvre** le glider au lieu
de le recopier). Ce notebook exige la même loi sur un **substrat totalement différent** :
l'univers fini des jeux 2×2 ordinaux de Robinson-Goforth. Si la loi ne tient que sur Life, ce n'est pas une loi.

Ici, le vérificateur c'est le solveur qui **reconnaît** qu'un chemin d'échanges élémentaires relie deux jeux.
Le constructeur, c'est la machine qui **produit** ce chemin — puis un **vérificateur séparé** qui vérifie,
indépendamment, que le chemin est valide **et minimal**. Le chemin produit est un **témoin construit et
vérifié indépendamment**, pas un certificat au sens formel : dans un univers fini et énumérable, la
minimalité se prouve par recherche exhaustive — c'est cette énumération, refaite de zéro, qui fait foi.

**Chiffres re-dérivés, jamais recopiés** : les chiffres structurels de l'espace de Robinson-Goforth
(75 ordres faibles, 5625 jeux à rangs, 576 chambres) sont **re-dérivés** dans chaque section ci-dessous
depuis zéro, pur `std` Rust — rien n'est recopié de GT-3b.

## 0. Configuration

Représentation héritée de GT-3b / GT-21 : un jeu ordinal 2×2 = une paire de tables de rangs
`(ligne, colonne)`, chacune un 4-uplet de rangs `{1..4}` lu en ligne-major — `t[0]` et `t[1]` sont la
première ligne, `t[2]` et `t[3]` la seconde. Le rang 4 est le meilleur payoff. Tout est **pur `std` (Rust)** :
l'univers tient en mémoire, aucune dépendance n'est nécessaire pour énumérer un espace fini.

In [2]:
// Metajeux-04 : le chemin minimal, temoin construit et verifie independamment.
// Representation : jeu ordinal 2x2 = paire de tables de rangs (ligne, colonne),
// chaque table un [u8; 4] lu en ligne-major, rang 4 = meilleur payoff.
// Pur std Rust : aucune dependance externe, l'univers fini tient en memoire.
println!("GameTheory 20 : chemin minimal (temoin construit et verifie independamment)");
println!("Representation : jeu = (table_ligne, table_colonne), rangs {{1..4}}, 4 = meilleur");
println!("Loi II : le constructeur PRODUIT le chemin, le verificateur separé le VERIFIE independamment");


GameTheory 20 : chemin minimal (temoin construit et verifie independamment)


Representation : jeu = (table_ligne, table_colonne), rangs {1..4}, 4 = meilleur


Loi II : le constructeur PRODUIT le chemin, le verificateur separé le VERIFIE independamment


**Lecture de la sortie committée** : la sortie se contente de trois lignes — les partis pris du
carnet, sans encore aucun dénombrement. D'abord la **représentation** : « jeu = (table_ligne,
table_colonne), rangs {1..4}, 4 = meilleur » — un jeu est une paire de tables de rangs, le rang
4 portant le meilleur payoff. Puis la **Loi II** : le constructeur **PRODUIT** le chemin,
et le **vérificateur séparé** le **vérifie indépendamment**. C'est ce double mouvement — produire le
chemin, puis le faire vérifier par un organe séparé — que les sections 3 et 4 soumettent à
l'épreuve sur l'univers fini des jeux.

## 1. L'univers fini, re-dérivé depuis zéro

Un **ordre faible** sur 4 cases = un classement avec ex æquo possibles. La clé canonique relabelle
les valeurs distinctes par ordre de première apparition : deux 4-uplets décrivant le même classement
partagent la même clé. Le nombre d'ordres faibles sur 4 éléments est le **nombre de Fubini** F₄ ;
le notebook le mesure au lieu de le réciter.

In [3]:
// Univers des ordres faibles et des chambres strictes : comptage des jeux a rangs, des chambres futures et des chambres strictes.
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
fn canonique(t: [u8; 4]) -> [u8; 4] {
    let mut vals: Vec<u8> = t.to_vec();
    vals.sort();
    vals.dedup();
    let mut res = [0u8; 4];
    for i in 0..4 {
        res[i] = vals.iter().position(|&x| x == t[i]).unwrap() as u8 + 1;
    }
    res
}
let mut ordres_faibles = BTreeSet::new();
for a in 1..=4u8 {
    for b in 1..=4u8 {
        for c in 1..=4u8 {
            for d in 1..=4u8 {
                let t = [a, b, c, d];
                ordres_faibles.insert(canonique(t));
            }
        }
    }
}
let stricts: Vec<[u8; 4]> = ordres_faibles.iter()
    .filter(|t| {
        let s: BTreeSet<u8> = t.iter().cloned().collect();
        s.len() == 4
    })
    .cloned()
    .collect();

println!("Ordres faibles sur 4 cases : {} (nombre de Fubini F4)", ordres_faibles.len());
println!("  stricts (futurs chambers)  : {}", stricts.len());
println!("  avec ex aequo (futurs murs) : {}", ordres_faibles.len() - stricts.len());
println!("Univers des jeux a rangs :  {} x {} = {}", ordres_faibles.len(), ordres_faibles.len(), ordres_faibles.len() * ordres_faibles.len());
println!("Univers des chambres strictes : {}", stricts.len() * stricts.len());


Ordres faibles sur 4 cases : 75 (nombre de Fubini F4)


  stricts (futurs chambers)  : 24


  avec ex aequo (futurs murs) : 51


Univers des jeux a rangs :  75 x 75 = 5625


Univers des chambres strictes : 576


**Lecture de la sortie committée** : **75** ordres faibles — c'est bien le nombre de Fubini F₄,
re-dérivé et non récité. Ils se partitionnent en **24 stricts** et **51 avec ex æquo**, d'où :
**5625 jeux à rangs** (l'univers complet, murs compris) et **576 chambres strictes** (l'univers de
Robinson-Goforth classique). Ces trois chiffres étaient RAPPORTÉS depuis la digestion ; ils sont
désormais VÉRIFIÉS par ce notebook, indépendamment de GT-3b.

### 1.2 Les jeux canoniques, encodés

Trois jeux classiques servent de bornes concrètes au constructeur. Encodage standard : pour un jeu
symétrique de dilemme 2×2 avec payoffs ordinaux `T > R > P > S` (défection tentante, punition mutuelle),
la table ligne lit `(R, S, T, P)` et la table colonne `(R, T, S, P)` — la transposition reflète le fait
que chaque joueur lit la même grille de son propre côté. Le Dilemme du Prisonnier suit `T > R > P > S`,
la Poule (Chicken) `T > R > S > P`, la Chasse au Cerf `R > T > P > S`.

In [4]:
// Les cinq bornes canoniques (identite, renversement, PD, Poule, Cerf) : representation et verification d'appartenance aux 576.
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
let pd: Jeu = ([3, 1, 4, 2], [3, 4, 1, 2]);
let poule: Jeu = ([3, 2, 4, 1], [3, 4, 2, 1]);
let cerf: Jeu = ([4, 1, 3, 2], [4, 3, 1, 2]);
let identite: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
let renverse: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
fn t4(t: [u8; 4]) -> String {
    format!("({}, {}, {}, {})", t[0], t[1], t[2], t[3])
}
fn jeu_repr(j: &Jeu) -> String {
    format!("({}, {})", t4(j.0), t4(j.1))
}
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}
fn py_list(v: &[String]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}

fn est_strict(jeu: &Jeu) -> bool {
    let s1: BTreeSet<u8> = jeu.0.iter().cloned().collect();
    let s2: BTreeSet<u8> = jeu.1.iter().cloned().collect();
    s1.len() == 4 && s2.len() == 4
}
fn afficher_jeu(jeu: &Jeu, nom: &str) {
    let (row, col) = jeu;
    println!("{}", nom);
    println!("  Ligne    |  {}  {} |   Colonne |  {}  {} |", row[0], row[1], col[0], col[1]);
    println!("           |  {}  {} |           |  {}  {} |", row[2], row[3], col[2], col[3]);
    println!("  strict (chambre) : {}", py_bool(est_strict(jeu)));
}
afficher_jeu(&pd, "Dilemme du Prisonnier");
afficher_jeu(&poule, "Poule (Chicken)");
afficher_jeu(&cerf, "Chasse au Cerf");
afficher_jeu(&identite, "Identite");
afficher_jeu(&renverse, "Renversement");
println!();
println!("Les cinq bornes canoniques sont des chambres strictes, toutes dans les 576.");


Dilemme du Prisonnier


  Ligne    |  3  1 |   Colonne |  3  4 |


           |  4  2 |           |  1  2 |


  strict (chambre) : True


Poule (Chicken)


  Ligne    |  3  2 |   Colonne |  3  4 |


... (11 outputs left out)

  Ligne    |  4  3 |   Colonne |  4  3 |


           |  2  1 |           |  2  1 |


  strict (chambre) : True


Les cinq bornes canoniques sont des chambres strictes, toutes dans les 576.


**Lecture de la sortie committée** : la sortie imprime les cinq jeux canoniques, chacun comme
une paire de tables de rangs — **Dilemme du Prisonnier** (Ligne 3 1 / 4 2, Colonne 3 4 / 1 2),
**Poule (Chicken)**, **Chasse au Cerf**, **Identité**, **Renversement** — et chaque table
emploie les quatre rangs {1, 2, 3, 4} exactement une fois, ce que récapitule le verdict «
strict (chambre) : True » répété cinq fois. Aux deux extrêmes, **Identité** répète la table 1 2
/ 3 4 des deux côtés et **Renversement** la table renversée 4 3 / 2 1 : les deux bornes sont
une permutation et son renversement. La dernière ligne conclut : « Les cinq bornes canoniques
sont des chambres strictes, toutes dans les 576. » — les cinq points de repère du cours sont
donc des sommets de l'univers strict dénombré à la section 1, et c'est entre eux que le
constructeur devra produire des chemins.

## 2. Les six swaps générateurs : le permutaèdre des chambres

Sur les chambres, la transformation élémentaire de Robinson-Goforth est l'**échange des valeurs
adjacentes k et k+1** dans une table : on permute les *cases* portant ces deux rangs. Trois échanges
sont possibles par table (k = 1, 2, 3), donc **six par jeu** — ce sont les « six swaps générateurs »
de la digestion. Le graphe obtenu sur les 24 ordres d'un côté est le **permutaèdre** S₄ ; sur les
576 chambres, son carré.

In [5]:
// Distances dans l'univers des chambres : diametre et repartition des distances depuis l'identite.
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
let pd: Jeu = ([3, 1, 4, 2], [3, 4, 1, 2]);
let poule: Jeu = ([3, 2, 4, 1], [3, 4, 2, 1]);
let cerf: Jeu = ([4, 1, 3, 2], [4, 3, 1, 2]);
let identite: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
let renverse: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
fn t4(t: [u8; 4]) -> String {
    format!("({}, {}, {}, {})", t[0], t[1], t[2], t[3])
}
fn jeu_repr(j: &Jeu) -> String {
    format!("({}, {})", t4(j.0), t4(j.1))
}
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}
fn py_list(v: &[String]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
fn canonique(t: [u8; 4]) -> [u8; 4] {
    let mut vals: Vec<u8> = t.to_vec();
    vals.sort();
    vals.dedup();
    let mut res = [0u8; 4];
    for i in 0..4 {
        res[i] = vals.iter().position(|&x| x == t[i]).unwrap() as u8 + 1;
    }
    res
}
let mut ordres_faibles = BTreeSet::new();
for a in 1..=4u8 {
    for b in 1..=4u8 {
        for c in 1..=4u8 {
            for d in 1..=4u8 {
                let t = [a, b, c, d];
                ordres_faibles.insert(canonique(t));
            }
        }
    }
}
let stricts: Vec<[u8; 4]> = ordres_faibles.iter()
    .filter(|t| {
        let s: BTreeSet<u8> = t.iter().cloned().collect();
        s.len() == 4
    })
    .cloned()
    .collect();
fn swap_valeurs_adjacentes(t: [u8; 4], k: u8) -> [u8; 4] {
    let mut res = t;
    if let (Some(a), Some(b)) = (t.iter().position(|&v| v == k), t.iter().position(|&v| v == k + 1)) {
        res.swap(a, b);
    }
    res
}
fn swap_jeu(jeu: &Jeu, cote: &str, k: u8) -> Jeu {
    match cote {
        "ligne" => (swap_valeurs_adjacentes(jeu.0, k), jeu.1),
        _ => (jeu.0, swap_valeurs_adjacentes(jeu.1, k)),
    }
}
let chambres: BTreeSet<Jeu> = {
    let mut s = BTreeSet::new();
    for r in &stricts {
        for c in &stricts {
            s.insert((*r, *c));
        }
    }
    s
};
let mut adj_chambres: BTreeMap<Jeu, Vec<Jeu>> = BTreeMap::new();
for jeu in &chambres {
    let mut voisins = Vec::new();
    for cote in &["ligne", "colonne"] {
        for k in 1..=3u8 {
            let t = if *cote == "ligne" { jeu.0 } else { jeu.1 };
            if t.contains(&k) && t.contains(&(k + 1)) {
                voisins.push(swap_jeu(jeu, cote, k));
            }
        }
    }
    adj_chambres.insert(*jeu, voisins);
}
fn bfs_complet<F>(depart: &Jeu, voisins_fn: F) -> BTreeMap<Jeu, usize>
where F: Fn(&Jeu) -> Vec<Jeu> {
    let mut dist = BTreeMap::new();
    let mut queue = VecDeque::new();
    dist.insert(*depart, 0);
    queue.push_back(*depart);
    while let Some(u) = queue.pop_front() {
        let d = dist[&u];
        for v in voisins_fn(&u) {
            if !dist.contains_key(&v) {
                dist.insert(v, d + 1);
                queue.push_back(v);
            }
        }
    }
    dist
}

let dist_identite = bfs_complet(&identite, |g| adj_chambres[g].clone());
println!("Chambres : {} | degre de chaque sommet : {}", chambres.len(), adj_chambres[&identite].len());
println!("Graphe des chambres : connexe = {} | diametre = {}",
    py_bool(dist_identite.len() == chambres.len()), dist_identite.values().max().unwrap());
let mut repartition: BTreeMap<usize, usize> = BTreeMap::new();
for d in dist_identite.values() {
    *repartition.entry(*d).or_insert(0) += 1;
}
println!("  repartition des distances : {:?}", repartition);


Chambres : 576 | degre de chaque sommet : 6


Graphe des chambres : connexe = True | diametre = 12


  repartition des distances : {0: 1, 1: 6, 2: 19, 3: 42, 4: 71, 5: 96, 6: 106, 7: 96, 8: 71, 9: 42, 10: 19, 11: 6, 12: 1}


**Lecture de la sortie committée** : le graphe des 576 chambres est **connexe**, de diamètre **12** —
six par côté, la distance d'un jeu étant la somme des distances de ses deux tables. La distribution
est **symétrique** (même effectif à d et 12−d) : l'involution qui renverse tous les rangs des deux
tables est une isométrie du permutaèdre. Le diamètre par table seule est 6, atteint par le renversement
`(1,2,3,4) → (4,3,2,1)` : chacune des six paires de positions doit être corrigée exactement une fois.

## 3. Le constructeur

C'est le geste que le vérificateur ne sait pas faire : étant donnés deux jeux **G** et **H**, **produire**
une suite d'échanges élémentaires qui mène de l'un à l'autre. Le constructeur parcourt le permutaèdre
en largeur depuis G, mémorise pour chaque sommet visité le move qui y a mené, puis **remonter la
chaîne des parents** depuis H exhume le chemin. Il ne connaît ni les jeux classiques, ni les noms —
il ne connaît que le graphe.

In [6]:
// Construction du chemin minimal PD -> Poule dans la grille des chambres.
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
let pd: Jeu = ([3, 1, 4, 2], [3, 4, 1, 2]);
let poule: Jeu = ([3, 2, 4, 1], [3, 4, 2, 1]);
let cerf: Jeu = ([4, 1, 3, 2], [4, 3, 1, 2]);
let identite: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
let renverse: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
fn t4(t: [u8; 4]) -> String {
    format!("({}, {}, {}, {})", t[0], t[1], t[2], t[3])
}
fn jeu_repr(j: &Jeu) -> String {
    format!("({}, {})", t4(j.0), t4(j.1))
}
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}
fn py_list(v: &[String]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
fn canonique(t: [u8; 4]) -> [u8; 4] {
    let mut vals: Vec<u8> = t.to_vec();
    vals.sort();
    vals.dedup();
    let mut res = [0u8; 4];
    for i in 0..4 {
        res[i] = vals.iter().position(|&x| x == t[i]).unwrap() as u8 + 1;
    }
    res
}
let mut ordres_faibles = BTreeSet::new();
for a in 1..=4u8 {
    for b in 1..=4u8 {
        for c in 1..=4u8 {
            for d in 1..=4u8 {
                let t = [a, b, c, d];
                ordres_faibles.insert(canonique(t));
            }
        }
    }
}
let stricts: Vec<[u8; 4]> = ordres_faibles.iter()
    .filter(|t| {
        let s: BTreeSet<u8> = t.iter().cloned().collect();
        s.len() == 4
    })
    .cloned()
    .collect();
fn swap_valeurs_adjacentes(t: [u8; 4], k: u8) -> [u8; 4] {
    let mut res = t;
    if let (Some(a), Some(b)) = (t.iter().position(|&v| v == k), t.iter().position(|&v| v == k + 1)) {
        res.swap(a, b);
    }
    res
}
fn swap_jeu(jeu: &Jeu, cote: &str, k: u8) -> Jeu {
    match cote {
        "ligne" => (swap_valeurs_adjacentes(jeu.0, k), jeu.1),
        _ => (jeu.0, swap_valeurs_adjacentes(jeu.1, k)),
    }
}
let chambres: BTreeSet<Jeu> = {
    let mut s = BTreeSet::new();
    for r in &stricts {
        for c in &stricts {
            s.insert((*r, *c));
        }
    }
    s
};
let mut adj_chambres: BTreeMap<Jeu, Vec<Jeu>> = BTreeMap::new();
for jeu in &chambres {
    let mut voisins = Vec::new();
    for cote in &["ligne", "colonne"] {
        for k in 1..=3u8 {
            let t = if *cote == "ligne" { jeu.0 } else { jeu.1 };
            if t.contains(&k) && t.contains(&(k + 1)) {
                voisins.push(swap_jeu(jeu, cote, k));
            }
        }
    }
    adj_chambres.insert(*jeu, voisins);
}
fn bfs_complet<F>(depart: &Jeu, voisins_fn: F) -> BTreeMap<Jeu, usize>
where F: Fn(&Jeu) -> Vec<Jeu> {
    let mut dist = BTreeMap::new();
    let mut queue = VecDeque::new();
    dist.insert(*depart, 0);
    queue.push_back(*depart);
    while let Some(u) = queue.pop_front() {
        let d = dist[&u];
        for v in voisins_fn(&u) {
            if !dist.contains_key(&v) {
                dist.insert(v, d + 1);
                queue.push_back(v);
            }
        }
    }
    dist
}
fn nommer_pas(u: &Jeu, v: &Jeu) -> String {
    if u.0 != v.0 {
        for k in 1..=3u8 {
            if swap_valeurs_adjacentes(u.0, k) == v.0 {
                return format!("swap ligne ({} <-> {})", k, k + 1);
            }
        }
    }
    if u.1 != v.1 {
        for k in 1..=3u8 {
            if swap_valeurs_adjacentes(u.1, k) == v.1 {
                return format!("swap colonne ({} <-> {})", k, k + 1);
            }
        }
    }
    "make_tie".to_string()
}
fn construire_chemin<F>(depart: &Jeu, arrivee: &Jeu, voisins_fn: F) -> Vec<(String, Jeu)>
where F: Fn(&Jeu) -> Vec<Jeu> {
    let mut parent: BTreeMap<Jeu, (Jeu, String)> = BTreeMap::new();
    let mut queue = VecDeque::new();
    let mut visited = BTreeSet::new();
    visited.insert(*depart);
    queue.push_back(*depart);
    while let Some(u) = queue.pop_front() {
        if u == *arrivee { break; }
        for v in voisins_fn(&u) {
            if !visited.contains(&v) {
                visited.insert(v);
                let label = nommer_pas(&u, &v);
                parent.insert(v, (u, label));
                queue.push_back(v);
            }
        }
    }
    let mut chemin = Vec::new();
    let mut cur = *arrivee;
    while cur != *depart {
        let (prev, label) = parent[&cur].clone();
        chemin.push((label, cur));
        cur = prev;
    }
    chemin.reverse();
    chemin
}

let chemin = construire_chemin(&pd, &poule, |g| adj_chambres[g].clone());
println!("Chemin minimal construit : Dilemme du Prisonnier -> Poule");
println!("  {} | {}   (depart)", t4(pd.0), t4(pd.1));
for (label, jeu) in &chemin {
    println!("  -- {} -->  {} | {}", label, t4(jeu.0), t4(jeu.1));
}
println!("Longueur du chemin produit : {} pas", chemin.len());


Chemin minimal construit : Dilemme du Prisonnier -> Poule


  (3, 1, 4, 2) | (3, 4, 1, 2)   (depart)


  -- swap ligne (1 <-> 2) -->  (3, 2, 4, 1) | (3, 4, 1, 2)


  -- swap colonne (1 <-> 2) -->  (3, 2, 4, 1) | (3, 4, 2, 1)


Longueur du chemin produit : 2 pas


**Lecture de la sortie committée** : le constructeur **produit** le chemin — deux pas exactement :
un échange `(1 ↔ 2)` côté ligne (la punition mutuelle P et la soumission S échangent leurs rangs) et
un échange `(1 ↔ 2)` côté colonne. Géométriquement, PD et Poule sont deux chambres voisines-à-deux-
pas dans le permutaèdre : ils ne diffèrent que par l'ordre des deux **mauvais** payoffs, le meilleur
(T) et le moyen (R) étant aux mêmes positions. Le dilemme et le jeu du poule : deux mondes séparés
par la seule question « quelle misère préférer ». La machine ne le savait pas — elle l'a construit.

In [7]:
// Chemin minimal PD -> Chasse au Cerf : le dilemme traverse le mur des chambres strictes.
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
let pd: Jeu = ([3, 1, 4, 2], [3, 4, 1, 2]);
let poule: Jeu = ([3, 2, 4, 1], [3, 4, 2, 1]);
let cerf: Jeu = ([4, 1, 3, 2], [4, 3, 1, 2]);
let identite: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
let renverse: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
fn t4(t: [u8; 4]) -> String {
    format!("({}, {}, {}, {})", t[0], t[1], t[2], t[3])
}
fn jeu_repr(j: &Jeu) -> String {
    format!("({}, {})", t4(j.0), t4(j.1))
}
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}
fn py_list(v: &[String]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
fn canonique(t: [u8; 4]) -> [u8; 4] {
    let mut vals: Vec<u8> = t.to_vec();
    vals.sort();
    vals.dedup();
    let mut res = [0u8; 4];
    for i in 0..4 {
        res[i] = vals.iter().position(|&x| x == t[i]).unwrap() as u8 + 1;
    }
    res
}
let mut ordres_faibles = BTreeSet::new();
for a in 1..=4u8 {
    for b in 1..=4u8 {
        for c in 1..=4u8 {
            for d in 1..=4u8 {
                let t = [a, b, c, d];
                ordres_faibles.insert(canonique(t));
            }
        }
    }
}
let stricts: Vec<[u8; 4]> = ordres_faibles.iter()
    .filter(|t| {
        let s: BTreeSet<u8> = t.iter().cloned().collect();
        s.len() == 4
    })
    .cloned()
    .collect();
fn swap_valeurs_adjacentes(t: [u8; 4], k: u8) -> [u8; 4] {
    let mut res = t;
    if let (Some(a), Some(b)) = (t.iter().position(|&v| v == k), t.iter().position(|&v| v == k + 1)) {
        res.swap(a, b);
    }
    res
}
fn swap_jeu(jeu: &Jeu, cote: &str, k: u8) -> Jeu {
    match cote {
        "ligne" => (swap_valeurs_adjacentes(jeu.0, k), jeu.1),
        _ => (jeu.0, swap_valeurs_adjacentes(jeu.1, k)),
    }
}
let chambres: BTreeSet<Jeu> = {
    let mut s = BTreeSet::new();
    for r in &stricts {
        for c in &stricts {
            s.insert((*r, *c));
        }
    }
    s
};
let mut adj_chambres: BTreeMap<Jeu, Vec<Jeu>> = BTreeMap::new();
for jeu in &chambres {
    let mut voisins = Vec::new();
    for cote in &["ligne", "colonne"] {
        for k in 1..=3u8 {
            let t = if *cote == "ligne" { jeu.0 } else { jeu.1 };
            if t.contains(&k) && t.contains(&(k + 1)) {
                voisins.push(swap_jeu(jeu, cote, k));
            }
        }
    }
    adj_chambres.insert(*jeu, voisins);
}
fn bfs_complet<F>(depart: &Jeu, voisins_fn: F) -> BTreeMap<Jeu, usize>
where F: Fn(&Jeu) -> Vec<Jeu> {
    let mut dist = BTreeMap::new();
    let mut queue = VecDeque::new();
    dist.insert(*depart, 0);
    queue.push_back(*depart);
    while let Some(u) = queue.pop_front() {
        let d = dist[&u];
        for v in voisins_fn(&u) {
            if !dist.contains_key(&v) {
                dist.insert(v, d + 1);
                queue.push_back(v);
            }
        }
    }
    dist
}
fn nommer_pas(u: &Jeu, v: &Jeu) -> String {
    if u.0 != v.0 {
        for k in 1..=3u8 {
            if swap_valeurs_adjacentes(u.0, k) == v.0 {
                return format!("swap ligne ({} <-> {})", k, k + 1);
            }
        }
    }
    if u.1 != v.1 {
        for k in 1..=3u8 {
            if swap_valeurs_adjacentes(u.1, k) == v.1 {
                return format!("swap colonne ({} <-> {})", k, k + 1);
            }
        }
    }
    "make_tie".to_string()
}
fn construire_chemin<F>(depart: &Jeu, arrivee: &Jeu, voisins_fn: F) -> Vec<(String, Jeu)>
where F: Fn(&Jeu) -> Vec<Jeu> {
    let mut parent: BTreeMap<Jeu, (Jeu, String)> = BTreeMap::new();
    let mut queue = VecDeque::new();
    let mut visited = BTreeSet::new();
    visited.insert(*depart);
    queue.push_back(*depart);
    while let Some(u) = queue.pop_front() {
        if u == *arrivee { break; }
        for v in voisins_fn(&u) {
            if !visited.contains(&v) {
                visited.insert(v);
                let label = nommer_pas(&u, &v);
                parent.insert(v, (u, label));
                queue.push_back(v);
            }
        }
    }
    let mut chemin = Vec::new();
    let mut cur = *arrivee;
    while cur != *depart {
        let (prev, label) = parent[&cur].clone();
        chemin.push((label, cur));
        cur = prev;
    }
    chemin.reverse();
    chemin
}

let chemin_long = construire_chemin(&identite, &renverse, |g| adj_chambres[g].clone());
println!("Chemin minimal construit : Identite -> Renversement (les deux antipodes du permutaedre)");
for (i, (label, _jeu)) in chemin_long.iter().enumerate() {
    println!("  pas {:>2} : {}", i + 1, label);
}
println!("Longueur : {} pas (le diametre exact du permutaedre S4 x S4)", chemin_long.len());
println!();
let chemin_cerf = construire_chemin(&pd, &cerf, |g| adj_chambres[g].clone());
let labels_cerf: Vec<String> = chemin_cerf.iter().map(|(l, _)| l.clone()).collect();
println!("Dilemme -> Chasse au Cerf : {} pas : {}", chemin_cerf.len(), py_list(&labels_cerf));


Chemin minimal construit : Identite -> Renversement (les deux antipodes du permutaedre)


  pas  1 : swap ligne (1 <-> 2)


  pas  2 : swap ligne (2 <-> 3)


  pas  3 : swap ligne (1 <-> 2)


  pas  4 : swap ligne (3 <-> 4)


  pas  5 : swap ligne (2 <-> 3)


... (5 outputs left out)

  pas 11 : swap colonne (2 <-> 3)


  pas 12 : swap colonne (1 <-> 2)


Longueur : 12 pas (le diametre exact du permutaedre S4 x S4)


Dilemme -> Chasse au Cerf : 2 pas : ['swap ligne (3 <-> 4)', 'swap colonne (3 <-> 4)']


**Lecture de la sortie committée** : l'antipode exige les **12 pas** du diamètre — chaque paire de positions des deux tables doit être corrigée exactement une fois, et le constructeur les exhume toutes. Ces douze pas se répartissent en deux blocs de six — swaps ligne aux pas 1 à 6, puis swaps colonne aux pas 7 à 12 : chaque table est renversée indépendamment, image directe de la structure produit S₄ × S₄. PD → Cerf est plus proche qu'il n'y paraît : deux pas, les échanges `(3 ↔ 4)` de chaque côté — seul l'ordre du **meilleur** payoff (coopération mutuelle R contre défection tentante T) change. La paire dilemme/cerf, structuralement opposée en théorie des jeux (dilemme social contre jeu de coordination), est géométriquement contiguë : c'est la lecture de Robinson-Goforth, reconstruite par la machine.


## 4. Le vérificateur séparé — le témoin

Produire un chemin ne suffit pas : encore faut-il le **vérifier indépendamment** — validité **et**
minimalité. Le témoin honnête exige un vérificateur qui ne réutilise **aucun état** du constructeur : il
re-dérive tout. Trois verdicts doivent être distingués — sinon le vérificateur n'est qu'un tampon :

- **VALIDE + MINIMAL** : chaque pas est un swap élémentaire, les extrémités sont exactes, et la
  longueur égale la distance exhaustive d(G, H) ;
- **VALIDE + NON MINIMAL** : le chemin marche mais un détour existe ;
- **INVALIDE** : au moins un pas n'est pas un move élémentaire (ou les extrémités mentent).

La minimalité se prouve par **recherche exhaustive** : BFS complet depuis G — l'univers est fini,
5625 ou 576 sommets, l'énumération totale EST la preuve qu'aucun raccourci n'existe.

In [8]:
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
let pd: Jeu = ([3, 1, 4, 2], [3, 4, 1, 2]);
let poule: Jeu = ([3, 2, 4, 1], [3, 4, 2, 1]);
let cerf: Jeu = ([4, 1, 3, 2], [4, 3, 1, 2]);
let identite: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
let renverse: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
fn t4(t: [u8; 4]) -> String {
    format!("({}, {}, {}, {})", t[0], t[1], t[2], t[3])
}
fn jeu_repr(j: &Jeu) -> String {
    format!("({}, {})", t4(j.0), t4(j.1))
}
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}
fn py_list(v: &[String]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
fn canonique(t: [u8; 4]) -> [u8; 4] {
    let mut vals: Vec<u8> = t.to_vec();
    vals.sort();
    vals.dedup();
    let mut res = [0u8; 4];
    for i in 0..4 {
        res[i] = vals.iter().position(|&x| x == t[i]).unwrap() as u8 + 1;
    }
    res
}
let mut ordres_faibles = BTreeSet::new();
for a in 1..=4u8 {
    for b in 1..=4u8 {
        for c in 1..=4u8 {
            for d in 1..=4u8 {
                let t = [a, b, c, d];
                ordres_faibles.insert(canonique(t));
            }
        }
    }
}
let stricts: Vec<[u8; 4]> = ordres_faibles.iter()
    .filter(|t| {
        let s: BTreeSet<u8> = t.iter().cloned().collect();
        s.len() == 4
    })
    .cloned()
    .collect();
fn swap_valeurs_adjacentes(t: [u8; 4], k: u8) -> [u8; 4] {
    let mut res = t;
    if let (Some(a), Some(b)) = (t.iter().position(|&v| v == k), t.iter().position(|&v| v == k + 1)) {
        res.swap(a, b);
    }
    res
}
fn swap_jeu(jeu: &Jeu, cote: &str, k: u8) -> Jeu {
    match cote {
        "ligne" => (swap_valeurs_adjacentes(jeu.0, k), jeu.1),
        _ => (jeu.0, swap_valeurs_adjacentes(jeu.1, k)),
    }
}
let chambres: BTreeSet<Jeu> = {
    let mut s = BTreeSet::new();
    for r in &stricts {
        for c in &stricts {
            s.insert((*r, *c));
        }
    }
    s
};
let mut adj_chambres: BTreeMap<Jeu, Vec<Jeu>> = BTreeMap::new();
for jeu in &chambres {
    let mut voisins = Vec::new();
    for cote in &["ligne", "colonne"] {
        for k in 1..=3u8 {
            let t = if *cote == "ligne" { jeu.0 } else { jeu.1 };
            if t.contains(&k) && t.contains(&(k + 1)) {
                voisins.push(swap_jeu(jeu, cote, k));
            }
        }
    }
    adj_chambres.insert(*jeu, voisins);
}
fn bfs_complet<F>(depart: &Jeu, voisins_fn: F) -> BTreeMap<Jeu, usize>
where F: Fn(&Jeu) -> Vec<Jeu> {
    let mut dist = BTreeMap::new();
    let mut queue = VecDeque::new();
    dist.insert(*depart, 0);
    queue.push_back(*depart);
    while let Some(u) = queue.pop_front() {
        let d = dist[&u];
        for v in voisins_fn(&u) {
            if !dist.contains_key(&v) {
                dist.insert(v, d + 1);
                queue.push_back(v);
            }
        }
    }
    dist
}
fn nommer_pas(u: &Jeu, v: &Jeu) -> String {
    if u.0 != v.0 {
        for k in 1..=3u8 {
            if swap_valeurs_adjacentes(u.0, k) == v.0 {
                return format!("swap ligne ({} <-> {})", k, k + 1);
            }
        }
    }
    if u.1 != v.1 {
        for k in 1..=3u8 {
            if swap_valeurs_adjacentes(u.1, k) == v.1 {
                return format!("swap colonne ({} <-> {})", k, k + 1);
            }
        }
    }
    "make_tie".to_string()
}
fn construire_chemin<F>(depart: &Jeu, arrivee: &Jeu, voisins_fn: F) -> Vec<(String, Jeu)>
where F: Fn(&Jeu) -> Vec<Jeu> {
    let mut parent: BTreeMap<Jeu, (Jeu, String)> = BTreeMap::new();
    let mut queue = VecDeque::new();
    let mut visited = BTreeSet::new();
    visited.insert(*depart);
    queue.push_back(*depart);
    while let Some(u) = queue.pop_front() {
        if u == *arrivee { break; }
        for v in voisins_fn(&u) {
            if !visited.contains(&v) {
                visited.insert(v);
                let label = nommer_pas(&u, &v);
                parent.insert(v, (u, label));
                queue.push_back(v);
            }
        }
    }
    let mut chemin = Vec::new();
    let mut cur = *arrivee;
    while cur != *depart {
        let (prev, label) = parent[&cur].clone();
        chemin.push((label, cur));
        cur = prev;
    }
    chemin.reverse();
    chemin
}
fn verifier_chemin<F>(depart: &Jeu, arrivee: &Jeu, chemin: &[(String, Jeu)], voisins_fn: F) -> String
where F: Fn(&Jeu) -> Vec<Jeu> {
    if chemin.is_empty() && depart == arrivee {
        return "VALIDE + MINIMAL : 0 pas".to_string();
    }
    let mut cur = *depart;
    for (_label, next) in chemin {
        let voisins = voisins_fn(&cur);
        if !voisins.contains(next) {
            return format!("INVALIDE : pas non elementaire vers {}", jeu_repr(next));
        }
        cur = *next;
    }
    if cur != *arrivee {
        return format!("INVALIDE : arrivee {} != {}", jeu_repr(&cur), jeu_repr(arrivee));
    }
    let dist = bfs_complet(depart, &voisins_fn);
    let d_reelle = dist.get(arrivee).copied().unwrap_or(usize::MAX);
    if chemin.len() == d_reelle {
        format!("VALIDE + MINIMAL : {} pas = distance exhaustive d(G,H)", chemin.len())
    } else {
        format!("VALIDE mais NON MINIMAL : {} pas, distance reelle {}", chemin.len(), d_reelle)
    }
}

// 1. le chemin du constructeur
let chemin = construire_chemin(&pd, &poule, |g| adj_chambres[g].clone());
let verdict1 = verifier_chemin(&pd, &poule, &chemin, |g| adj_chambres[g].clone());
println!("1. Le chemin du constructeur (PD -> Poule) :");
println!("   {}", verdict1);
// 2. le meme chemin rallonge d'un aller-retour sur le dernier swap
let mut etendu = chemin.clone();
let dernier_etat = chemin.last().unwrap().1;
let avant_dernier = chemin[chemin.len() - 2].1;
let (mut cote_d, mut k_d) = ("", 0u8);
'cherche: for cote in ["ligne", "colonne"] {
    for k in 1..=3u8 {
        if swap_jeu(&avant_dernier, cote, k) == dernier_etat {
            cote_d = cote;
            k_d = k;
            break 'cherche;
        }
    }
}
let aller = swap_jeu(&poule, cote_d, k_d);
etendu.push((nommer_pas(&poule, &aller), aller));
let retour = swap_jeu(&aller, cote_d, k_d);
etendu.push((nommer_pas(&aller, &retour), retour));
let verdict2 = verifier_chemin(&pd, &poule, &etendu, |g| adj_chambres[g].clone());
println!("2. Le meme chemin rallonge d'un aller-retour :");
println!("   {}", verdict2);
// 3. un 'chemin' en un pas vers un jeu non adjacent
let faux_chemin = vec![("saut direct".to_string(), cerf)];
let verdict3 = verifier_chemin(&pd, &cerf, &faux_chemin, |g| adj_chambres[g].clone());
println!("3. Un 'chemin' en un pas vers un jeu non adjacent :");
println!("   {}", verdict3);


1. Le chemin du constructeur (PD -> Poule) :


   VALIDE + MINIMAL : 2 pas = distance exhaustive d(G,H)


2. Le meme chemin rallonge d'un aller-retour :


   VALIDE mais NON MINIMAL : 4 pas, distance reelle 2


3. Un 'chemin' en un pas vers un jeu non adjacent :


   INVALIDE : pas non elementaire vers ((4, 1, 3, 2), (4, 3, 1, 2))


**Lecture de la sortie committée** : les trois verdicts tombent, distincts. Le chemin du constructeur
est **VALIDE + MINIMAL** (2 pas = distance exhaustive). Le même chemin avec un aller-retour greffé est
**VALIDE mais NON MINIMAL** — le vérificateur voit le détour. Et le saut direct PD → Cerf est
**INVALIDE** : un pas non élémentaire, parce que les deux chambres ne sont pas adjacentes. Le
vérificateur n'est pas un tampon : il rend trois réponses différentes sur trois défauts différents.
C'est la séparation constructeur/vérificateur de la Loi II — le même geste que la preuve `by decide`
du translateur de Life, ici sur le substrat fini des jeux.

### 4.2 La non-unicité : compter les géodésiques

Un chemin minimal n'est **jamais seul**. Les géodésiques entre deux jeux forment les « mondes
minimaux » — autant de façons distinctes de transformer l'un en l'autre sans un pas de trop. Le
comptage se fait par le DAG des couches BFS : un sommet de couche d reçoit ses chemins de ses
prédécesseurs de couche d−1. Pour le renversement d'une table (l'antipode du permutaèdre S₄),
la combinatoire des groupes de Coxeter prédit que le nombre de mots réduits de l'élément le plus
long w₀ de S₄ vaut 16 — une prédiction externe que la mesure peut contredire.

In [9]:
// Comptage des geodesiques (nombres de chemins minimaux) entre les jeux de l'univers des chambres.
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
let pd: Jeu = ([3, 1, 4, 2], [3, 4, 1, 2]);
let poule: Jeu = ([3, 2, 4, 1], [3, 4, 2, 1]);
let cerf: Jeu = ([4, 1, 3, 2], [4, 3, 1, 2]);
let identite: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
let renverse: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
fn t4(t: [u8; 4]) -> String {
    format!("({}, {}, {}, {})", t[0], t[1], t[2], t[3])
}
fn jeu_repr(j: &Jeu) -> String {
    format!("({}, {})", t4(j.0), t4(j.1))
}
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}
fn py_list(v: &[String]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
fn canonique(t: [u8; 4]) -> [u8; 4] {
    let mut vals: Vec<u8> = t.to_vec();
    vals.sort();
    vals.dedup();
    let mut res = [0u8; 4];
    for i in 0..4 {
        res[i] = vals.iter().position(|&x| x == t[i]).unwrap() as u8 + 1;
    }
    res
}
let mut ordres_faibles = BTreeSet::new();
for a in 1..=4u8 {
    for b in 1..=4u8 {
        for c in 1..=4u8 {
            for d in 1..=4u8 {
                let t = [a, b, c, d];
                ordres_faibles.insert(canonique(t));
            }
        }
    }
}
let stricts: Vec<[u8; 4]> = ordres_faibles.iter()
    .filter(|t| {
        let s: BTreeSet<u8> = t.iter().cloned().collect();
        s.len() == 4
    })
    .cloned()
    .collect();
fn swap_valeurs_adjacentes(t: [u8; 4], k: u8) -> [u8; 4] {
    let mut res = t;
    if let (Some(a), Some(b)) = (t.iter().position(|&v| v == k), t.iter().position(|&v| v == k + 1)) {
        res.swap(a, b);
    }
    res
}
fn swap_jeu(jeu: &Jeu, cote: &str, k: u8) -> Jeu {
    match cote {
        "ligne" => (swap_valeurs_adjacentes(jeu.0, k), jeu.1),
        _ => (jeu.0, swap_valeurs_adjacentes(jeu.1, k)),
    }
}
let chambres: BTreeSet<Jeu> = {
    let mut s = BTreeSet::new();
    for r in &stricts {
        for c in &stricts {
            s.insert((*r, *c));
        }
    }
    s
};
let mut adj_chambres: BTreeMap<Jeu, Vec<Jeu>> = BTreeMap::new();
for jeu in &chambres {
    let mut voisins = Vec::new();
    for cote in &["ligne", "colonne"] {
        for k in 1..=3u8 {
            let t = if *cote == "ligne" { jeu.0 } else { jeu.1 };
            if t.contains(&k) && t.contains(&(k + 1)) {
                voisins.push(swap_jeu(jeu, cote, k));
            }
        }
    }
    adj_chambres.insert(*jeu, voisins);
}
fn bfs_complet<F>(depart: &Jeu, voisins_fn: F) -> BTreeMap<Jeu, usize>
where F: Fn(&Jeu) -> Vec<Jeu> {
    let mut dist = BTreeMap::new();
    let mut queue = VecDeque::new();
    dist.insert(*depart, 0);
    queue.push_back(*depart);
    while let Some(u) = queue.pop_front() {
        let d = dist[&u];
        for v in voisins_fn(&u) {
            if !dist.contains_key(&v) {
                dist.insert(v, d + 1);
                queue.push_back(v);
            }
        }
    }
    dist
}

fn compter_geodesiques<F>(depart: &Jeu, arrivee: &Jeu, voisins_fn: F) -> usize
where F: Fn(&Jeu) -> Vec<Jeu> {
    let dist_depart = bfs_complet(depart, &voisins_fn);
    let dist_arrivee = bfs_complet(arrivee, &voisins_fn);
    let d = match dist_depart.get(arrivee) {
        Some(&d) => d,
        None => return 0,
    };
    let mut dag: BTreeMap<Jeu, Vec<Jeu>> = BTreeMap::new();
    for (u, &du) in &dist_depart {
        if let Some(&dv) = dist_arrivee.get(u) {
            if du + dv == d {
                let voisins: Vec<Jeu> = voisins_fn(u).into_iter()
                    .filter(|v| dist_depart.contains_key(v)
                        && dist_arrivee.contains_key(v)
                        && dist_depart[v] + dist_arrivee[v] == d
                        && dist_depart[v] == du + 1)
                    .collect();
                dag.insert(*u, voisins);
            }
        }
    }
    let mut count: BTreeMap<Jeu, usize> = BTreeMap::new();
    count.insert(*depart, 1);
    let mut sorted: Vec<(&Jeu, &usize)> = dist_depart.iter().collect();
    sorted.sort_by_key(|(_, d)| **d);
    for (u, _) in &sorted {
        if let Some(&c) = count.get(u) {
            if let Some(voisins) = dag.get(u) {
                for v in voisins {
                    *count.entry(*v).or_insert(0) += c;
                }
            }
        }
    }
    count.get(arrivee).copied().unwrap_or(0)
}
let t_identite: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
let t_renverse: Jeu = ([4, 3, 2, 1], [1, 2, 3, 4]);
let geo_une_table = compter_geodesiques(&t_identite, &t_renverse, |g| adj_chambres[g].clone());
println!("Geodesiques (1,2,3,4) -> (4,3,2,1) sur une table : {} (mots reduits de w0 dans S4 : 16 attendus)", geo_une_table);
let geo_pd_poule = compter_geodesiques(&pd, &poule, |g| adj_chambres[g].clone());
println!("Geodesiques PD -> Poule sur les {} chambres : {}", chambres.len(), geo_pd_poule);
let geo_jeu = compter_geodesiques(&identite, &renverse, |g| adj_chambres[g].clone());
println!("Geodesiques Identite -> Renversement (diametre 12) : {}", geo_jeu);


Geodesiques (1,2,3,4) -> (4,3,2,1) sur une table : 16 (mots reduits de w0 dans S4 : 16 attendus)


Geodesiques PD -> Poule sur les 576 chambres : 2


Geodesiques Identite -> Renversement (diametre 12) : 236544


**Lecture de la sortie committée** : la mesure donne **16** géodésiques pour le renversement d'une
table — exactement le nombre de mots réduits de w₀ dans S₄ que prédit la théorie des groupes de
Coxeter. La mesure rejoint la combinatoire connue : le DAG des couches ne calcule rien d'autre que
les expressions réduites du plus long élément. Pour le jeu complet, la composition n'est **pas** un
simple produit : les pas des deux côtés s'**entrelacent**. PD → Poule (un pas par table) offre
**2** géodésiques — l'ordre des deux pas ; l'antipode complet en offre **236 544 = 16 × 16 × 924**,
les mots réduits par côté multipliés par les C(12,6) = 924 entrelacements des deux séquences de six
pas. Le chemin produit au §3 n'était qu'un tirage parmi cette flore.

## 5. L'univers à rangs complet : traverser les murs

Le permutaèdre ne couvre que les chambres. L'univers à rangs complet (5625) inclut les **murs** —
les jeux à égalités, strates de codimension ≥ 1 de GT-3b. Deux moves élémentaires s'ajoutent aux
swaps : **make_tie** (coalescence des valeurs k et k+1 : la chambre descend sur un mur) et son
inverse **break_tie** (résoudre l'ex æquo dans un sens ou l'autre : traverser). En graphe non
orienté, les deux directions d'un même edge suffisent : coalescer depuis chacune des chambres
adjacentes énumère toutes les résolutions du mur.

La question que seul l'univers complet peut trancher : **les murs sont-ils des raccourcis ?**
Un chemin qui passe par un mur peut-il être plus court que la géodésique des chambres ? L'intuition
suggère que oui — un swap dans un mur déplace un bloc fusionné d'un seul coup. La mesure va
décider, et elle ne suit pas l'intuition.

In [10]:
// Repartition des distances dans l'univers complet des jeux a rangs.
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
let pd: Jeu = ([3, 1, 4, 2], [3, 4, 1, 2]);
let poule: Jeu = ([3, 2, 4, 1], [3, 4, 2, 1]);
let cerf: Jeu = ([4, 1, 3, 2], [4, 3, 1, 2]);
let identite: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
let renverse: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
fn t4(t: [u8; 4]) -> String {
    format!("({}, {}, {}, {})", t[0], t[1], t[2], t[3])
}
fn jeu_repr(j: &Jeu) -> String {
    format!("({}, {})", t4(j.0), t4(j.1))
}
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}
fn py_list(v: &[String]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
fn canonique(t: [u8; 4]) -> [u8; 4] {
    let mut vals: Vec<u8> = t.to_vec();
    vals.sort();
    vals.dedup();
    let mut res = [0u8; 4];
    for i in 0..4 {
        res[i] = vals.iter().position(|&x| x == t[i]).unwrap() as u8 + 1;
    }
    res
}
let mut ordres_faibles = BTreeSet::new();
for a in 1..=4u8 {
    for b in 1..=4u8 {
        for c in 1..=4u8 {
            for d in 1..=4u8 {
                let t = [a, b, c, d];
                ordres_faibles.insert(canonique(t));
            }
        }
    }
}
let stricts: Vec<[u8; 4]> = ordres_faibles.iter()
    .filter(|t| {
        let s: BTreeSet<u8> = t.iter().cloned().collect();
        s.len() == 4
    })
    .cloned()
    .collect();
fn swap_valeurs_adjacentes(t: [u8; 4], k: u8) -> [u8; 4] {
    let mut res = t;
    if let (Some(a), Some(b)) = (t.iter().position(|&v| v == k), t.iter().position(|&v| v == k + 1)) {
        res.swap(a, b);
    }
    res
}
fn swap_jeu(jeu: &Jeu, cote: &str, k: u8) -> Jeu {
    match cote {
        "ligne" => (swap_valeurs_adjacentes(jeu.0, k), jeu.1),
        _ => (jeu.0, swap_valeurs_adjacentes(jeu.1, k)),
    }
}
fn bfs_complet<F>(depart: &Jeu, voisins_fn: F) -> BTreeMap<Jeu, usize>
where F: Fn(&Jeu) -> Vec<Jeu> {
    let mut dist = BTreeMap::new();
    let mut queue = VecDeque::new();
    dist.insert(*depart, 0);
    queue.push_back(*depart);
    while let Some(u) = queue.pop_front() {
        let d = dist[&u];
        for v in voisins_fn(&u) {
            if !dist.contains_key(&v) {
                dist.insert(v, d + 1);
                queue.push_back(v);
            }
        }
    }
    dist
}
fn faire_le_tie(t: [u8; 4], k: u8) -> [u8; 4] {
    let mut res = [0u8; 4];
    for i in 0..4 {
        res[i] = if t[i] == k + 1 { k } else { t[i] };
    }
    canonique(res)
}
fn voisins_complets(jeu: &Jeu) -> Vec<Jeu> {
    let mut res = Vec::new();
    for cote in &["ligne", "colonne"] {
        let t = if *cote == "ligne" { jeu.0 } else { jeu.1 };
        let valeurs: BTreeSet<u8> = t.iter().cloned().collect();
        for k in 1..=3u8 {
            if valeurs.contains(&k) && valeurs.contains(&(k + 1)) {
                res.push(swap_jeu(jeu, cote, k));
                let t2 = faire_le_tie(t, k);
                if t2 != t {
                    let new_jeu = if *cote == "ligne" { (t2, jeu.1) } else { (jeu.0, t2) };
                    res.push(new_jeu);
                }
            }
        }
    }
    res
}

let univers: BTreeSet<Jeu> = {
    let mut s = BTreeSet::new();
    for r in &ordres_faibles {
        for c in &ordres_faibles {
            s.insert((*r, *c));
        }
    }
    s
};
let dist_univers = bfs_complet(&identite, voisins_complets);
println!("Univers complet : {} sommets, atteints : {} (connexe : {} )",
    univers.len(), dist_univers.len(), py_bool(dist_univers.len() == univers.len()));
println!("Diametre de l'univers complet : {}", dist_univers.values().max().unwrap());
println!("  (a comparer au diametre 12 du seul permutaedre des chambres)");
let mut rep: BTreeMap<usize, usize> = BTreeMap::new();
for d in dist_univers.values() {
    *rep.entry(*d).or_insert(0) += 1;
}
println!("  repartition : {:?}", rep);


Univers complet : 5625 sommets, atteints : 5625 (connexe : True )


Diametre de l'univers complet : 12


  (a comparer au diametre 12 du seul permutaedre des chambres)


  repartition : {0: 1, 1: 12, 2: 68, 3: 240, 4: 578, 5: 990, 6: 1232, 7: 1128, 8: 785, 9: 402, 10: 149, 11: 36, 12: 4}


**Lecture de la sortie committée** : l'univers complet est **connexe** — coalescences et traversées relient les 5625 — mais son diamètre **reste 12**, identique au permutaèdre des chambres, avec les mêmes 4 antipodes. La répartition culmine à 1232 sommets à la distance 6 — le milieu exact de l'intervalle [0, 12] — la distribution garde sa masse maximale en couche 6 et retombe exactement sur les mêmes extrêmes. L'intuition « les murs raccourcissent » vient de se heurter à un premier fait : au niveau du diamètre, rien ne bouge. Reste à savoir si les murs raccourcissent **certaines** distances locales — c'est l'objet du §5.2, qui le décide par énumération exhaustive.


In [11]:
// Univers complet : chemin PD -> Renversement, nombre de murs traverses, et verification independante du chemin.
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
let pd: Jeu = ([3, 1, 4, 2], [3, 4, 1, 2]);
let poule: Jeu = ([3, 2, 4, 1], [3, 4, 2, 1]);
let cerf: Jeu = ([4, 1, 3, 2], [4, 3, 1, 2]);
let identite: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
let renverse: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
fn t4(t: [u8; 4]) -> String {
    format!("({}, {}, {}, {})", t[0], t[1], t[2], t[3])
}
fn jeu_repr(j: &Jeu) -> String {
    format!("({}, {})", t4(j.0), t4(j.1))
}
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}
fn py_list(v: &[String]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
fn canonique(t: [u8; 4]) -> [u8; 4] {
    let mut vals: Vec<u8> = t.to_vec();
    vals.sort();
    vals.dedup();
    let mut res = [0u8; 4];
    for i in 0..4 {
        res[i] = vals.iter().position(|&x| x == t[i]).unwrap() as u8 + 1;
    }
    res
}
let mut ordres_faibles = BTreeSet::new();
for a in 1..=4u8 {
    for b in 1..=4u8 {
        for c in 1..=4u8 {
            for d in 1..=4u8 {
                let t = [a, b, c, d];
                ordres_faibles.insert(canonique(t));
            }
        }
    }
}
let stricts: Vec<[u8; 4]> = ordres_faibles.iter()
    .filter(|t| {
        let s: BTreeSet<u8> = t.iter().cloned().collect();
        s.len() == 4
    })
    .cloned()
    .collect();
fn swap_valeurs_adjacentes(t: [u8; 4], k: u8) -> [u8; 4] {
    let mut res = t;
    if let (Some(a), Some(b)) = (t.iter().position(|&v| v == k), t.iter().position(|&v| v == k + 1)) {
        res.swap(a, b);
    }
    res
}
fn swap_jeu(jeu: &Jeu, cote: &str, k: u8) -> Jeu {
    match cote {
        "ligne" => (swap_valeurs_adjacentes(jeu.0, k), jeu.1),
        _ => (jeu.0, swap_valeurs_adjacentes(jeu.1, k)),
    }
}
let chambres: BTreeSet<Jeu> = {
    let mut s = BTreeSet::new();
    for r in &stricts {
        for c in &stricts {
            s.insert((*r, *c));
        }
    }
    s
};
let mut adj_chambres: BTreeMap<Jeu, Vec<Jeu>> = BTreeMap::new();
for jeu in &chambres {
    let mut voisins = Vec::new();
    for cote in &["ligne", "colonne"] {
        for k in 1..=3u8 {
            let t = if *cote == "ligne" { jeu.0 } else { jeu.1 };
            if t.contains(&k) && t.contains(&(k + 1)) {
                voisins.push(swap_jeu(jeu, cote, k));
            }
        }
    }
    adj_chambres.insert(*jeu, voisins);
}
fn bfs_complet<F>(depart: &Jeu, voisins_fn: F) -> BTreeMap<Jeu, usize>
where F: Fn(&Jeu) -> Vec<Jeu> {
    let mut dist = BTreeMap::new();
    let mut queue = VecDeque::new();
    dist.insert(*depart, 0);
    queue.push_back(*depart);
    while let Some(u) = queue.pop_front() {
        let d = dist[&u];
        for v in voisins_fn(&u) {
            if !dist.contains_key(&v) {
                dist.insert(v, d + 1);
                queue.push_back(v);
            }
        }
    }
    dist
}
fn nommer_pas(u: &Jeu, v: &Jeu) -> String {
    if u.0 != v.0 {
        for k in 1..=3u8 {
            if swap_valeurs_adjacentes(u.0, k) == v.0 {
                return format!("swap ligne ({} <-> {})", k, k + 1);
            }
        }
    }
    if u.1 != v.1 {
        for k in 1..=3u8 {
            if swap_valeurs_adjacentes(u.1, k) == v.1 {
                return format!("swap colonne ({} <-> {})", k, k + 1);
            }
        }
    }
    "make_tie".to_string()
}
fn construire_chemin<F>(depart: &Jeu, arrivee: &Jeu, voisins_fn: F) -> Vec<(String, Jeu)>
where F: Fn(&Jeu) -> Vec<Jeu> {
    let mut parent: BTreeMap<Jeu, (Jeu, String)> = BTreeMap::new();
    let mut queue = VecDeque::new();
    let mut visited = BTreeSet::new();
    visited.insert(*depart);
    queue.push_back(*depart);
    while let Some(u) = queue.pop_front() {
        if u == *arrivee { break; }
        for v in voisins_fn(&u) {
            if !visited.contains(&v) {
                visited.insert(v);
                let label = nommer_pas(&u, &v);
                parent.insert(v, (u, label));
                queue.push_back(v);
            }
        }
    }
    let mut chemin = Vec::new();
    let mut cur = *arrivee;
    while cur != *depart {
        let (prev, label) = parent[&cur].clone();
        chemin.push((label, cur));
        cur = prev;
    }
    chemin.reverse();
    chemin
}
fn verifier_chemin<F>(depart: &Jeu, arrivee: &Jeu, chemin: &[(String, Jeu)], voisins_fn: F) -> String
where F: Fn(&Jeu) -> Vec<Jeu> {
    if chemin.is_empty() && depart == arrivee {
        return "VALIDE + MINIMAL : 0 pas".to_string();
    }
    let mut cur = *depart;
    for (_label, next) in chemin {
        let voisins = voisins_fn(&cur);
        if !voisins.contains(next) {
            return format!("INVALIDE : pas non elementaire vers {}", jeu_repr(next));
        }
        cur = *next;
    }
    if cur != *arrivee {
        return format!("INVALIDE : arrivee {} != {}", jeu_repr(&cur), jeu_repr(arrivee));
    }
    let dist = bfs_complet(depart, &voisins_fn);
    let d_reelle = dist.get(arrivee).copied().unwrap_or(usize::MAX);
    if chemin.len() == d_reelle {
        format!("VALIDE + MINIMAL : {} pas = distance exhaustive d(G,H)", chemin.len())
    } else {
        format!("VALIDE mais NON MINIMAL : {} pas, distance reelle {}", chemin.len(), d_reelle)
    }
}
fn faire_le_tie(t: [u8; 4], k: u8) -> [u8; 4] {
    let mut res = [0u8; 4];
    for i in 0..4 {
        res[i] = if t[i] == k + 1 { k } else { t[i] };
    }
    canonique(res)
}
fn voisins_complets(jeu: &Jeu) -> Vec<Jeu> {
    let mut res = Vec::new();
    for cote in &["ligne", "colonne"] {
        let t = if *cote == "ligne" { jeu.0 } else { jeu.1 };
        let valeurs: BTreeSet<u8> = t.iter().cloned().collect();
        for k in 1..=3u8 {
            if valeurs.contains(&k) && valeurs.contains(&(k + 1)) {
                res.push(swap_jeu(jeu, cote, k));
                let t2 = faire_le_tie(t, k);
                if t2 != t {
                    let new_jeu = if *cote == "ligne" { (t2, jeu.1) } else { (jeu.0, t2) };
                    res.push(new_jeu);
                }
            }
        }
    }
    res
}

fn voisins_cote_perm(t: [u8; 4]) -> Vec<[u8; 4]> {
    let mut res = Vec::new();
    for k in 1..=3u8 {
        if t.contains(&k) && t.contains(&(k + 1)) {
            res.push(swap_valeurs_adjacentes(t, k));
        }
    }
    res
}
fn voisins_cote_full(t: [u8; 4]) -> Vec<[u8; 4]> {
    let mut res = Vec::new();
    for k in 1..=3u8 {
        if t.contains(&k) && t.contains(&(k + 1)) {
            res.push(swap_valeurs_adjacentes(t, k));
            let t2 = faire_le_tie(t, k);
            if t2 != t {
                res.push(t2);
            }
        }
    }
    res
}
fn bfs_cote<F>(depart: [u8; 4], voisins_fn: F) -> BTreeMap<[u8; 4], usize>
where F: Fn([u8; 4]) -> Vec<[u8; 4]> {
    let mut dist = BTreeMap::new();
    let mut queue = VecDeque::new();
    dist.insert(depart, 0);
    queue.push_back(depart);
    while let Some(u) = queue.pop_front() {
        let d = dist[&u];
        for v in voisins_fn(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, d + 1);
                queue.push_back(v);
            }
        }
    }
    dist
}
let mut ecarts = 0;
for a in &stricts {
    let d_perm = bfs_cote(*a, voisins_cote_perm);
    let d_full = bfs_cote(*a, voisins_cote_full);
    for b in &stricts {
        let dp = d_perm.get(b).copied().unwrap_or(usize::MAX);
        let df = d_full.get(b).copied().unwrap_or(usize::MAX);
        if dp != df {
            ecarts += 1;
        }
    }
}
println!("Paires d'ordres stricts comparees (24 x 24) : 576");
println!("Paires ou l'espace complet (murs autorises) est PLUS COURT que le permutaedre : {}", ecarts);
println!();
println!("Par structure produit, le verdict couvre les {} x {} = {} paires de chambres :",
    chambres.len(), chambres.len(), chambres.len() * chambres.len());
println!("  les murs ne raccourcissent AUCUNE distance inter-chambres.");
let chemin_full = construire_chemin(&pd, &renverse, voisins_complets);
let murs_traverses = chemin_full.iter()
    .filter(|(_, cur)| {
        let s1: BTreeSet<u8> = cur.0.iter().cloned().collect();
        let s2: BTreeSet<u8> = cur.1.iter().cloned().collect();
        s1.len() < 4 || s2.len() < 4
    })
    .count();
let chemin_pur = construire_chemin(&pd, &renverse, |g| adj_chambres[g].clone());
println!();
println!("PD -> Renversement : univers complet {} pas dont {} sur un mur ; chambres seules {} pas",
    chemin_full.len(), murs_traverses, chemin_pur.len());
let v = verifier_chemin(&pd, &renverse, &chemin_full, voisins_complets);
println!("Verificateur (univers complet) : {}", v);


Paires d'ordres stricts comparees (24 x 24) : 576


Paires ou l'espace complet (murs autorises) est PLUS COURT que le permutaedre : 0


Par structure produit, le verdict couvre les 576 x 576 = 331776 paires de chambres :


  les murs ne raccourcissent AUCUNE distance inter-chambres.


PD -> Renversement : univers complet 5 pas dont 0 sur un mur ; chambres seules 5 pas


Verificateur (univers complet) : VALIDE + MINIMAL : 5 pas = distance exhaustive d(G,H)


**Lecture de la sortie committée** : la décision est **nulle et exhaustive** — sur les 576 paires
d'ordres stricts comparées côté par côté, **aucune** n'est plus courte par les murs ; et parce que
les générateurs ne touchent qu'un côté à la fois (structure produit), ce verdict par côté couvre
les 331 776 paires de chambres sans exception. Le mécanisme se voit sur le cas concret : le
constructeur lâché dans l'univers complet produit un chemin qui **ne pose jamais le pied sur un
mur** — la même longueur, 5 pas, que dans les chambres seules, et le vérificateur séparé l'atteste
minimal dans l'univers élargi. Descendre sur un mur coûte un pas, en sortir en coûte un autre,
et le swap qui déplace le bloc fusionné ne récupère jamais plus que ces deux pas investis.
Les murs de GT-3b ne sont donc pas des tunnels : ce sont des strates que la géométrie des chambres
regarde traverser sans jamais y gagner un pas. Un résultat négatif — mais prouvé par énumération
complète, et c'est lui qui donne sa valeur au témoin : la minimalité ne dit pas seulement
« pas de détour », elle dit « pas de détour, même par les murs ».

## 6. L'epaisseur des murs — les chambres en forme extensive

Les cinq sections precedentes repondent a une question de **classement** : quelle chambre contient
cette matrice, et quel est le chemin minimal d'une chambre a l'autre. En forme normale, le mur entre
deux chambres est une frontiere de nomenclature : on le franchit en **changeant de matrice**, pas en
jouant. La question extensive deplace le sujet : *quel chemin causal et informationnel rend une
issue atteignable a matrice fixee ?* Le mur cesse d'etre une etiquette pour devenir un objet
**traversable-sous-conditions** — ce qui le franchit est une promesse ou une menace **croyable**,
et la croyabilite est exactement ce que l'arbre sait calculer (sous-jeux, induction a rebours) ;
le mur `w = l` y est un separateur de codimension 1, pas un nom.

**Le dispositif.** Avant le jeu de base, le joueur Ligne choisit un **engagement observable** :
se taire, promettre (se lier a Cooperer — la force acquise en se privant d'une option, Schelling),
ou menacer (programmer une punition si Colonne defectionne). Chaque variante devient un arbre
explicite : noeuds de decision, ensembles d'information (singletons ici, l'engagement etant
observe — la partition est imprimee avec l'arbre), feuilles valorisees. La croyabilite n'est
jamais assertee : elle se **decide par induction a rebours au sous-jeu de punition**.

**Unites.** Les issues du jeu de base gardent les rangs {1..4} de la chambre ; l'issue de punition
porte des utilites cardinales (0,0), sous le plancher ordinal des rangs — n'importe quelle
utilite coherente avec l'ordre de la chambre donne le meme verdict (robustesse ordinale).

In [12]:
// Arbre de jeu SPE : le sous-jeu de punition montre que la menace CHOISIE est incroyable (verdict par induction).
#[derive(Clone, Debug)]
enum Noeud {
    Feuille { u: (i32, i32), nom: String },
    Decision { joueur: usize, enfants: Vec<(String, Noeud)>, info: String },
}
fn feuille(u1: i32, u2: i32, nom: &str) -> Noeud {
    Noeud::Feuille { u: (u1, u2), nom: nom.to_string() }
}
fn decision(joueur: usize, enfants: Vec<(String, Noeud)>, info: &str) -> Noeud {
    Noeud::Decision { joueur, enfants, info: info.to_string() }
}
fn u_j(u: (i32, i32), joueur: usize) -> i32 {
    if joueur == 0 { u.0 } else { u.1 }
}
fn valeur_spe(noeud: &Noeud) -> (i32, i32) {
    match noeud {
        Noeud::Feuille { u, .. } => *u,
        Noeud::Decision { joueur, enfants, .. } => {
            let mut best: Option<(i32, i32)> = None;
            for (_action, enfant) in enfants {
                let v = valeur_spe(enfant);
                if best.is_none() || u_j(v, *joueur) > u_j(best.unwrap(), *joueur) {
                    best = Some(v);
                }
            }
            best.unwrap()
        }
    }
}
fn chemin_spe(noeud: &Noeud, trace: &[String]) -> Vec<(Vec<String>, String, (i32, i32))> {
    match noeud {
        Noeud::Feuille { u, nom } => {
            vec![(trace.to_vec(), nom.clone(), *u)]
        }
        Noeud::Decision { joueur, enfants, .. } => {
            let best_u = valeur_spe(noeud);
            let mut sorties = Vec::new();
            for (action, enfant) in enfants {
                let v = valeur_spe(enfant);
                if u_j(v, *joueur) == u_j(best_u, *joueur) {
                    let mut new_trace = trace.to_vec();
                    new_trace.push(format!("J{}:{}", joueur + 1, action));
                    sorties.extend(chemin_spe(enfant, &new_trace));
                }
            }
            sorties
        }
    }
}
fn partition_information(noeud: &Noeud, acc: &mut Vec<String>) {
    match noeud {
        Noeud::Decision { info, enfants, .. } => {
            acc.push(info.clone());
            for (_a, enfant) in enfants {
                partition_information(enfant, acc);
            }
        }
        Noeud::Feuille { .. } => {}
    }
}

println!("Moteur : induction a rebours sur arbre fini, ensembles d'information explicites.");
println!("Test de coherence sur un mini-arbre a menace choisie :");
let test = decision(0, vec![
    ("laisser".to_string(), feuille(1, 4, "laisser (S,T)")),
    ("punir".to_string(), feuille(0, 0, "punir (0,0)")),
], "test:punition");
let u = valeur_spe(&test);
let chemins = chemin_spe(&test, &[]);
let choix = chemins[0].0.last().cloned().unwrap();
println!("  sous-jeu de punition : valeur SPE = {:?}, action retenue = {}", u, choix);
println!("  -> punir rapporte 0 < laisser 1 au punisseur : la menace CHOISIE est INCROYABLE");
println!("  (le verdict vient de l'induction, pas d'une assertion).");


Moteur : induction a rebours sur arbre fini, ensembles d'information explicites.


Test de coherence sur un mini-arbre a menace choisie :


  sous-jeu de punition : valeur SPE = (1, 4), action retenue = J1:laisser


  -> punir rapporte 0 < laisser 1 au punisseur : la menace CHOISIE est INCROYABLE


  (le verdict vient de l'induction, pas d'une assertion).


**Lecture de la sortie commitee** : sur le sous-jeu de punition seul, l'induction retient
*laisser* (1 > 0 pour le punisseur) — une menace **laissee au choix** de son auteur ne survit pas
a l'induction a rebours. C'est le critere operational de la section : la meme menace, **programmee**
(branche retiree de l'arbre), devient croyable par construction. Reste a mesurer ce que chaque
forme d'engagement fait au jeu de base, chambre par chambre.

In [13]:
// Modes de transition vers le play cooperatif : atteignabilite du point cooperatif par mode.
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
let pd: Jeu = ([3, 1, 4, 2], [3, 4, 1, 2]);
let poule: Jeu = ([3, 2, 4, 1], [3, 4, 2, 1]);
let cerf: Jeu = ([4, 1, 3, 2], [4, 3, 1, 2]);
let identite: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
let renverse: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
#[derive(Clone, Debug)]
enum Noeud {
    Feuille { u: (i32, i32), nom: String },
    Decision { joueur: usize, enfants: Vec<(String, Noeud)>, info: String },
}
fn feuille(u1: i32, u2: i32, nom: &str) -> Noeud {
    Noeud::Feuille { u: (u1, u2), nom: nom.to_string() }
}
fn decision(joueur: usize, enfants: Vec<(String, Noeud)>, info: &str) -> Noeud {
    Noeud::Decision { joueur, enfants, info: info.to_string() }
}
fn u_j(u: (i32, i32), joueur: usize) -> i32 {
    if joueur == 0 { u.0 } else { u.1 }
}
fn valeur_spe(noeud: &Noeud) -> (i32, i32) {
    match noeud {
        Noeud::Feuille { u, .. } => *u,
        Noeud::Decision { joueur, enfants, .. } => {
            let mut best: Option<(i32, i32)> = None;
            for (_action, enfant) in enfants {
                let v = valeur_spe(enfant);
                if best.is_none() || u_j(v, *joueur) > u_j(best.unwrap(), *joueur) {
                    best = Some(v);
                }
            }
            best.unwrap()
        }
    }
}
fn chemin_spe(noeud: &Noeud, trace: &[String]) -> Vec<(Vec<String>, String, (i32, i32))> {
    match noeud {
        Noeud::Feuille { u, nom } => {
            vec![(trace.to_vec(), nom.clone(), *u)]
        }
        Noeud::Decision { joueur, enfants, .. } => {
            let best_u = valeur_spe(noeud);
            let mut sorties = Vec::new();
            for (action, enfant) in enfants {
                let v = valeur_spe(enfant);
                if u_j(v, *joueur) == u_j(best_u, *joueur) {
                    let mut new_trace = trace.to_vec();
                    new_trace.push(format!("J{}:{}", joueur + 1, action));
                    sorties.extend(chemin_spe(enfant, &new_trace));
                }
            }
            sorties
        }
    }
}
fn partition_information(noeud: &Noeud, acc: &mut Vec<String>) {
    match noeud {
        Noeud::Decision { info, enfants, .. } => {
            acc.push(info.clone());
            for (_a, enfant) in enfants {
                partition_information(enfant, acc);
            }
        }
        Noeud::Feuille { .. } => {}
    }
}

fn extraire_cellule(jeu: &Jeu, i: usize, j: usize) -> (i32, i32) {
    let (tl, tc) = jeu;
    (tl[2 * i + j] as i32, tc[2 * i + j] as i32)
}
fn py_infos(v: &[String]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
let pd_cellules: BTreeMap<(usize, usize), (i32, i32)> = {
    let mut m = BTreeMap::new();
    for i in 0..2usize {
        for j in 0..2usize {
            m.insert((i, j), extraire_cellule(&pd, i, j));
        }
    }
    m
};
fn arbre_dilemme(mode: &str, p: &BTreeMap<(usize, usize), (i32, i32)>) -> Noeud {
    match mode {
        "silence" => {
            let suite = |j2: usize| -> (i32, i32) {
                let meilleure_j1 = if p[&(0, j2)].0 > p[&(1, j2)].0 { 0 } else { 1 };
                p[&(meilleure_j1, j2)]
            };
            decision(1, vec![
                ("C".to_string(), feuille(suite(0).0, suite(0).1, "(BR,C)")),
                ("D".to_string(), feuille(suite(1).0, suite(1).1, "(BR,D)")),
            ], "silence:J2-jeu-de-base")
        }
        "promesse" => {
            decision(1, vec![
                ("C".to_string(), feuille(p[&(0, 0)].0, p[&(0, 0)].1, "(C,C)")),
                ("D".to_string(), feuille(p[&(0, 1)].0, p[&(0, 1)].1, "(C,D)")),
            ], "promesse:J2-observe-C-force")
        }
        "menace_choisie" => {
            let punition = decision(0, vec![
                ("laisser".to_string(), feuille(p[&(0, 1)].0, p[&(0, 1)].1, "(C,D)-laissee")),
                ("punir".to_string(), feuille(0, 0, "(C,D)-punie")),
            ], "menace:J1-choisit");
            decision(1, vec![
                ("C".to_string(), feuille(p[&(0, 0)].0, p[&(0, 0)].1, "(C,C)")),
                ("D".to_string(), punition),
            ], "menace-choisie:J2-observe")
        }
        _ => {
            decision(1, vec![
                ("C".to_string(), feuille(p[&(0, 0)].0, p[&(0, 0)].1, "(C,C)")),
                ("D".to_string(), feuille(0, 0, "(C,D)-punie-automatiquement")),
            ], "menace-programmee:J2-observe")
        }
    }
}
let mut verdicts: Vec<(String, (i32, i32))> = Vec::new();
for mode in &["silence", "promesse", "menace_choisie", "menace_programmee"] {
    let arbre = arbre_dilemme(mode, &pd_cellules);
    let u = valeur_spe(&arbre);
    let chemins = chemin_spe(&arbre, &[]);
    let mut infos = Vec::new();
    partition_information(&arbre, &mut infos);
    verdicts.push((mode.to_string(), u));
    println!("--- Dilemme / engagement = {}", mode);
    println!("    ensembles d'information : {}", py_infos(&infos));
    for (trace, nom, uu) in &chemins {
        println!("    chemin SPE {:38} issue {:32} utilites {:?}", trace.join(" -> "), nom, uu);
    }
    println!("    valeur SPE : {:?}", u);
}
println!();
let cooperative = pd_cellules[&(0, 0)];
println!("Play cooperatif (C,C) = {:?} (utilites du Dilemme)", cooperative);
for (mode, u) in &verdicts {
    let atteint = if *u == cooperative { "ATTEINTE " } else { "inatteignable" };
    println!("  {:18} -> {:?} : transition vers le play cooperatif : {}", mode, u, atteint);
}
let _ = ();


--- Dilemme / engagement = silence


    ensembles d'information : ['silence:J2-jeu-de-base']


    chemin SPE J2:D                                   issue (BR,D)                           utilites (2, 2)


    valeur SPE : (2, 2)


--- Dilemme / engagement = promesse


    ensembles d'information : ['promesse:J2-observe-C-force']


... (11 outputs left out)

  silence            -> (2, 2) : transition vers le play cooperatif : inatteignable


  promesse           -> (1, 4) : transition vers le play cooperatif : inatteignable


  menace_choisie     -> (1, 4) : transition vers le play cooperatif : inatteignable


  menace_programmee  -> (3, 3) : transition vers le play cooperatif : ATTEINTE 


**Lecture de la sortie commitee** : a matrice **fixee** dans la chambre Dilemme, la transition vers
le play cooperatif (3,3) est **inatteignable** au silence (l'equilibre de meilleure reponse reste
la defection mutuelle), **inatteignable** par la promesse seule (Colonne exploite la main liee :
(1,4)), **inatteignable** par la menace choisie (l'induction du sous-jeu de punition la retourne :
laisser domine punir, et Colonne anticipe) — et **atteinte** par la menace programmee observable
(3,3), la punition ne s'executant jamais : c'est une **dissuasion**. Le mur n'a pas ete franchi
dans l'atlas des 576 chambres — la matrice n'a pas bouge d'un rang — mais le **play** a traverse
ce que la forme normale tenait pour une frontiere. C'est l'epaisseur du mur : non pas une distance
dans le permutaedre, mais un ensemble d'arbres d'engagement capables ou non de conduire a l'issue.

In [14]:
// Silence et promesse auto-executoire : la MEME promesse, deux chambres, deux epaisseurs de mur.
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
let pd: Jeu = ([3, 1, 4, 2], [3, 4, 1, 2]);
let poule: Jeu = ([3, 2, 4, 1], [3, 4, 2, 1]);
let cerf: Jeu = ([4, 1, 3, 2], [4, 3, 1, 2]);
let identite: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
let renverse: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
#[derive(Clone, Debug)]
enum Noeud {
    Feuille { u: (i32, i32), nom: String },
    Decision { joueur: usize, enfants: Vec<(String, Noeud)>, info: String },
}
fn feuille(u1: i32, u2: i32, nom: &str) -> Noeud {
    Noeud::Feuille { u: (u1, u2), nom: nom.to_string() }
}
fn decision(joueur: usize, enfants: Vec<(String, Noeud)>, info: &str) -> Noeud {
    Noeud::Decision { joueur, enfants, info: info.to_string() }
}
fn u_j(u: (i32, i32), joueur: usize) -> i32 {
    if joueur == 0 { u.0 } else { u.1 }
}
fn valeur_spe(noeud: &Noeud) -> (i32, i32) {
    match noeud {
        Noeud::Feuille { u, .. } => *u,
        Noeud::Decision { joueur, enfants, .. } => {
            let mut best: Option<(i32, i32)> = None;
            for (_action, enfant) in enfants {
                let v = valeur_spe(enfant);
                if best.is_none() || u_j(v, *joueur) > u_j(best.unwrap(), *joueur) {
                    best = Some(v);
                }
            }
            best.unwrap()
        }
    }
}
fn chemin_spe(noeud: &Noeud, trace: &[String]) -> Vec<(Vec<String>, String, (i32, i32))> {
    match noeud {
        Noeud::Feuille { u, nom } => {
            vec![(trace.to_vec(), nom.clone(), *u)]
        }
        Noeud::Decision { joueur, enfants, .. } => {
            let best_u = valeur_spe(noeud);
            let mut sorties = Vec::new();
            for (action, enfant) in enfants {
                let v = valeur_spe(enfant);
                if u_j(v, *joueur) == u_j(best_u, *joueur) {
                    let mut new_trace = trace.to_vec();
                    new_trace.push(format!("J{}:{}", joueur + 1, action));
                    sorties.extend(chemin_spe(enfant, &new_trace));
                }
            }
            sorties
        }
    }
}
fn partition_information(noeud: &Noeud, acc: &mut Vec<String>) {
    match noeud {
        Noeud::Decision { info, enfants, .. } => {
            acc.push(info.clone());
            for (_a, enfant) in enfants {
                partition_information(enfant, acc);
            }
        }
        Noeud::Feuille { .. } => {}
    }
}

fn extraire_cellule(jeu: &Jeu, i: usize, j: usize) -> (i32, i32) {
    let (tl, tc) = jeu;
    (tl[2 * i + j] as i32, tc[2 * i + j] as i32)
}
fn py_infos(v: &[String]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
let cerf_cellules: BTreeMap<(usize, usize), (i32, i32)> = {
    let mut m = BTreeMap::new();
    for i in 0..2usize {
        for j in 0..2usize {
            m.insert((i, j), extraire_cellule(&cerf, i, j));
        }
    }
    m
};
fn arbre_cerf_promesse(cc: &BTreeMap<(usize, usize), (i32, i32)>) -> Noeud {
    decision(1, vec![
        ("C".to_string(), feuille(cc[&(0, 0)].0, cc[&(0, 0)].1, "(C,C)")),
        ("D".to_string(), feuille(cc[&(0, 1)].0, cc[&(0, 1)].1, "(C,D)")),
    ], "cerf:J2-observe-C-force")
}
fn arbre_cerf_silence(cc: &BTreeMap<(usize, usize), (i32, i32)>) -> Noeud {
    let suite = |j2: usize| -> (i32, i32) {
        let meilleure_j1 = if cc[&(0, j2)].0 > cc[&(1, j2)].0 { 0 } else { 1 };
        cc[&(meilleure_j1, j2)]
    };
    decision(1, vec![
        ("C".to_string(), feuille(suite(0).0, suite(0).1, "(BR,C)")),
        ("D".to_string(), feuille(suite(1).0, suite(1).1, "(BR,D)")),
    ], "cerf-silence:J2-jeu-de-base")
}
for (mode, arbre) in &[("silence", arbre_cerf_silence(&cerf_cellules)), ("promesse observable", arbre_cerf_promesse(&cerf_cellules))] {
    let u = valeur_spe(arbre);
    println!("--- Cerf / engagement = {}", mode);
    let mut infos = Vec::new();
    partition_information(arbre, &mut infos);
    println!("    ensembles d'information : {}", py_infos(&infos));
    for (trace, nom, uu) in &chemin_spe(arbre, &[]) {
        println!("    chemin SPE {:38} issue {:10} utilites {:?}", trace.join(" -> "), nom, uu);
    }
    println!("    valeur SPE : {:?}", u);
}
println!();
println!("Play cooperatif (C,C) du Cerf = {:?}", cerf_cellules[&(0, 0)]);
println!("Au silence, le Cerf coordonne aussi sur (C,C) par meilleures reponses croisees :");
println!("la promesse y est AUTO-EXECUTOIRE (C est deja meilleure reponse a C), le mur du Cerf");
println!("etait poreux ; celle du Dilemme exigeait l'automatisation de la punition.");
println!("Memoriser : la MEME promesse, deux chambres, deux epaisseurs de mur.");


--- Cerf / engagement = silence


    ensembles d'information : ['cerf-silence:J2-jeu-de-base']


    chemin SPE J2:C                                   issue (BR,C)     utilites (4, 4)


    valeur SPE : (4, 4)


--- Cerf / engagement = promesse observable


    ensembles d'information : ['cerf:J2-observe-C-force']


... (3 outputs left out)

Au silence, le Cerf coordonne aussi sur (C,C) par meilleures reponses croisees :


la promesse y est AUTO-EXECUTOIRE (C est deja meilleure reponse a C), le mur du Cerf


etait poreux ; celle du Dilemme exigeait l'automatisation de la punition.


Memoriser : la MEME promesse, deux chambres, deux epaisseurs de mur.


**Lecture de la sortie commitee** : la promesse liee-au-cerf est **auto-executoire** — dans la
chambre Cerf, cooperer est deja la meilleure reponse a cooperer, l'engagement ne retire qu'un risk
de coordination, le play cooperatif etait atteignable sans lui. Dans la chambre Dilemme, la meme
promesse livree a Colonne ne vaut que le rang du dindon de la farce (1,4). **La forme extensive ne
deplace pas la chambre ; elle mesure la resistance du mur a chaque engin de traversree** — et cette
resistance est une propriete ordinale de la chambre (T>R>P>S vs R>T>P>S), donc un invariant de
l'atlas Robinson-Goforth lui-meme : deux chambres adjacentes peuvent porter des epaisseurs tres
differentes pour le meme engagement.

## 7. Ce que la loi autorise à dire — et ce qu'elle n'autorise pas

**Ce qui est acquis** : le passage du vérificateur au constructeur est fait sur un **second substrat**.
Étant donnés deux jeux, la machine **produit** la suite d'opérations élémentaires qui transforme
l'un en l'autre, et un vérificateur séparé **vérifie indépendamment** validité et minimalité par
énumération exhaustive de l'univers. Les trois verdicts du vérificateur sont distincts et opposables ; le
comptage des géodésiques rejoint la combinatoire de Coxeter (16 mots réduits de w₀ par table,
composés en 236 544 par entrelacement) ; et le résultat négatif du §5 est **prouvé par énumération
complète** : les murs ne raccourcissent aucune distance inter-chambres — le diamètre reste 12.
La loi n'était pas une spécificité de Life, et le substrat des jeux a livré une géométrie que le
seul permutaèdre ne montrait pas.

**Ce qui n'est PAS acquis** — les frontières honnêtes :

- **Ordinal 2×2 uniquement**. L'espace des jeux à payoffs cardinaux, ou même 3×3 ordinaux,
  n'est ni énuméré ni borné ici ; la vérification par exhaustivité meurt avec la finitude.
- **Le chemin est une géométrie, pas une dynamique**. Rien ne dit qu'un processus d'adaptation
  suiverait une géodésique ; les swaps ne sont pas des coups que des joueurs jouent.
- **La minimalité est relative au jeu de moves**. Avec swaps sur les deux tables, la distance
  d'un jeu est la somme des distances par table (12 pour le jeu, 6 par table) : le verdict
  énonce « minimal pour CES générateurs », jamais « le plus court possible dans l'absolu ».
- **Pas de jambe Lean dans ce grain**. La vérification est l'énumération exhaustive Rust ; la
  formalisation `by decide` de la validité d'un pas est la variante naturelle suivante — la
  forme d'itération (`-b`, `-c`) que suit cette série.

**Sources** — Robinson & Goforth (1978), *The Topology of the 2×2 Games* (cité, RAPPORTÉ) ;
décomposition de Bruns-Kimmich, chambres / murs / `make_tie` : GT-3b ; le précédent
constructeur de ce type : le translateur Life (GOL-06). Chiffres structurels re-dérivés ici :
75, 576, 5625, diamètre 12 dans les deux univers, résultat du non-raccourci, géodésiques
16 / 2 / 236 544 (VÉRIFIÉS par ce notebook).

## Exercices

Les trois exercices utilisent le constructeur, le vérificateur et le compteur de géodésiques
définis plus haut. Le notebook s'exécute de bout en bout même sans les compléter.

### Exercice 1 — Le chemin retour

Construire le chemin **Poule → Dilemme** et le soumettre au vérificateur séparé. Vérifier que le
constructeur produit bien le même nombre de pas qu'à l'aller — et expliquer pourquoi la distance est
symétrique.

In [15]:
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
// Exercice 1 : chemin retour Poule -> Dilemme, construit puis verifie independamment.
// Etape 1 : chemin_retour = construire_chemin(&poule, &pd, |g| adj_chambres[g].clone())
// Etape 2 : verdict = verifier_chemin(&poule, &pd, &chemin_retour, |g| adj_chambres[g].clone())
// Indice : la symetrie vient de ce que chaque swap est sa propre reciproque.
let chemin_retour: Option<Vec<(String, Jeu)>> = None; // TODO etudiant
println!("Exercice 1 : a completer (chemin retour Poule -> Dilemme)");


Exercice 1 : a completer (chemin retour Poule -> Dilemme)


### Exercice 2 — Les géodésiques d'une autre paire

Compter les géodésiques **Dilemme → Chasse au Cerf** dans l'univers complet (murs autorisés),
et comparer au compte dans les chambres seules. Le mur crée-t-il de nouveaux mondes minimaux ?

In [16]:
// Exercice 2 : geodesiques PD -> Cerf, univers complet vs chambres.
// Etape 1 : nb_complet = compter_geodesiques(&pd, &cerf, voisins_complets)
// Etape 2 : comparer a compter_geodesiques(&pd, &cerf, |g| adj_chambres[g].clone())
// Indice : si la distance change entre les deux univers, le compte change aussi.
let nb_geodesiques_complet: Option<usize> = None; // TODO etudiant
let nb_geodesiques_chambres: Option<usize> = None; // TODO etudiant
println!("Exercice 2 : a completer (geodesiques PD -> Cerf, deux univers)");


Exercice 2 : a completer (geodesiques PD -> Cerf, deux univers)


### Exercice 3 — Un couple antipodal de l'univers complet

Trouver un jeu **H** à distance maximale du Dilemme dans l'univers complet (5625), construire le
chemin, et le soumettre au vérificateur séparé. Indice : la distribution des distances du §5.1 dit
quelle couche chercher.

In [17]:
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
// Exercice 3 : antipode du Dilemme dans l'univers complet (5625).
// Etape 1 : recalculer bfs_complet(&pd, voisins_complets), chercher les sommets de distance max.
// Etape 2 : construire_chemin + verifier_chemin vers l'un d'eux.
// Indice : l'antipode n'est pas unique -- la couche maximale compte plusieurs sommets.
let antipode_pd: Option<Jeu> = None; // TODO etudiant : le jeu (row, col) a distance maximale de PD
println!("Exercice 3 : a completer (antipode du Dilemme dans l'univers complet)");


Exercice 3 : a completer (antipode du Dilemme dans l'univers complet)


### Exercice 4 — La Poule en forme extensive

La chambre Poule (T>R>S>P) a deux equilibres purs en strategies : (C,D) et (D,C). Un eleve de
Robinson-Goforth propose : « Ligne s'engage OBSERVABLEMENT a faire le mort (se lier a D) ; Colonne,
voyant la roue abandonnee, fait pareil par prudence... non, s'ecarte ! »

Construisez l'arbre `arbre_poule_engagement_D` (Ligne liee a D, observable, Colonne repond),
executeriez l'induction, et comparez a (C,C). Le mur de la Poule est-il poreux a cet engin-là ?
Quel engagement rendrait (C,C) atteignable — et pourquoi la reponse est-elle plus difficile
que pour le Cerf ?

In [18]:
type Jeu = ([u8; 4], [u8; 4]);
use std::collections::{BTreeMap, BTreeSet, VecDeque};
#[derive(Clone, Debug)]
enum Noeud {
    Feuille { u: (i32, i32), nom: String },
    Decision { joueur: usize, enfants: Vec<(String, Noeud)>, info: String },
}
fn feuille(u1: i32, u2: i32, nom: &str) -> Noeud {
    Noeud::Feuille { u: (u1, u2), nom: nom.to_string() }
}
fn decision(joueur: usize, enfants: Vec<(String, Noeud)>, info: &str) -> Noeud {
    Noeud::Decision { joueur, enfants, info: info.to_string() }
}
fn u_j(u: (i32, i32), joueur: usize) -> i32 {
    if joueur == 0 { u.0 } else { u.1 }
}
fn valeur_spe(noeud: &Noeud) -> (i32, i32) {
    match noeud {
        Noeud::Feuille { u, .. } => *u,
        Noeud::Decision { joueur, enfants, .. } => {
            let mut best: Option<(i32, i32)> = None;
            for (_action, enfant) in enfants {
                let v = valeur_spe(enfant);
                if best.is_none() || u_j(v, *joueur) > u_j(best.unwrap(), *joueur) {
                    best = Some(v);
                }
            }
            best.unwrap()
        }
    }
}
fn chemin_spe(noeud: &Noeud, trace: &[String]) -> Vec<(Vec<String>, String, (i32, i32))> {
    match noeud {
        Noeud::Feuille { u, nom } => {
            vec![(trace.to_vec(), nom.clone(), *u)]
        }
        Noeud::Decision { joueur, enfants, .. } => {
            let best_u = valeur_spe(noeud);
            let mut sorties = Vec::new();
            for (action, enfant) in enfants {
                let v = valeur_spe(enfant);
                if u_j(v, *joueur) == u_j(best_u, *joueur) {
                    let mut new_trace = trace.to_vec();
                    new_trace.push(format!("J{}:{}", joueur + 1, action));
                    sorties.extend(chemin_spe(enfant, &new_trace));
                }
            }
            sorties
        }
    }
}
fn partition_information(noeud: &Noeud, acc: &mut Vec<String>) {
    match noeud {
        Noeud::Decision { info, enfants, .. } => {
            acc.push(info.clone());
            for (_a, enfant) in enfants {
                partition_information(enfant, acc);
            }
        }
        Noeud::Feuille { .. } => {}
    }
}
// Exercice 4 : Poule en forme extensive — Ligne liee a D, observable.
// Indice : reutilisez extraire_cellule(&poule, i, j), decision(), valeur_spe(), chemin_spe().
// Etape 1 : construire poule_cellules (BTreeMap des utilites de la Poule).
// Etape 2 : construire l'arbre (branche D de Ligne retiree, Colonne observe).
// Etape 3 : verdict — valeur SPE vs (C,C), chemin d'equilibre.
let poule_cellules: Option<BTreeMap<(usize, usize), (i32, i32)>> = None; // TODO etudiant
let arbre_poule_engagement_d: Option<Noeud> = None; // TODO etudiant
println!("Exercice a completer : verdict de l'engagement D observable sur la Poule.");


Exercice a completer : verdict de l'engagement D observable sur la Poule.


## À retenir

- **La loi II est franchie sur un second substrat** : le constructeur *produit* la suite d'échanges élémentaires qui transforme un jeu en un autre, et un **vérificateur séparé** — qui ne réutilise aucun état du constructeur — *vérifie indépendamment* validité ET minimalité par énumération exhaustive de l'univers fini.
- **Le comptage des géodésiques rejoint la combinatoire de Coxeter** : 16 géodésiques pour le renversement d'une table, exactement le nombre de mots réduits de l'élément le plus long `w₀` de `S₄` ; le DAG des couches BFS ne calcule rien d'autre que ces expressions réduites.
- **Résultat négatif prouvé par énumération complète** : les murs (jeux à égalités) ne raccourcissent AUCUNE distance inter-chambres — le diamètre reste 12, identique au permutaèdre des chambres. C'est un fait prouvé, pas l'absence de contre-exemple trouvé.

## Vérifiez votre compréhension

1. Pourquoi le vérificateur doit-il être *séparé* du constructeur (ne réutiliser aucun de ses états) ? Que permet chacun des trois verdicts distincts (VALIDE+MINIMAL / VALIDE+NON MINIMAL / INVALIDE) d'exclure ?
2. Le comptage donne 16 géodésiques pour l'antipode d'une table. Quelle prédiction externe de la théorie des groupes de Coxeter rejoint-il, et qu'aurait signifié un écart ?
3. En quoi « les murs ne raccourcissent rien » est-il un résultat *prouvé* et non une absence de contre-exemple ? Quel rôle joue la structure produit `S₄ × S₄` dans la portée de l'énumération ?

<details>
<summary>Éléments de réponse</summary>

1. Un constructeur ne peut pas se contrôler lui-même : le BFS qui trouve le chemin s'appuie sur ses propres structures (parents, table d'adjacence), et s'il vérifiait son propre résultat, une erreur de ces structures serait reproduite à l'identique — le contrôle ne distinguerait pas un chemin valide d'un chemin fabriqué par la même erreur. Le vérificateur du §4 ne réutilise rien : il re-dérive l'élémentarité de chaque pas contre le graphe, contrôle les extrémités, puis refait un BFS complet pour la distance. Les trois verdicts séparent trois défauts. VALIDE+MINIMAL exclut à la fois le pas non élémentaire et le détour : la sortie imprime `VALIDE + MINIMAL : 2 pas = distance exhaustive d(G,H)` pour le chemin du constructeur, et le même verdict sur l'univers complet (`5 pas`). VALIDE mais NON MINIMAL isole le cas « le chemin marche mais un plus court existe » : `4 pas, distance reelle 2`. INVALIDE rejette un pas qui n'est pas un move élémentaire ou des extrémités fausses : `pas non elementaire vers ((4, 1, 3, 2), (4, 3, 1, 2))` pour le saut PD → Cerf, deux chambres non adjacentes. Un vérificateur qui ne saurait dire que « valide » serait un tampon ; c'est le troisième verdict qui lui donne les dents.
2. La prédiction externe est le nombre de mots réduits de l'élément le plus long `w₀` de `S₄` : **16**. La sortie le mesure et le nomme — `Geodesiques (1,2,3,4) -> (4,3,2,1) sur une table : 16 (mots reduits de w0 dans S4 : 16 attendus)`. Le DAG des couches ne calcule rien d'autre que ces expressions réduites : une géodésique est une suite de swaps adjacents sans pas inutile, donc un mot réduit. L'intérêt de la prédiction est qu'elle est **externe** : elle vient de la combinatoire de Coxeter, pas d'une ligne du carnet, et la mesure peut la contredire. Un écart aurait donc signalé que l'instrument ne mesure pas ce qu'il prétend — générateurs qui ne sont pas les transpositions adjacentes standard de `S₄`, ou comptage par couches faux — et c'est la mesure qui aurait été réfutée. La mesure tombe juste, et la composition au jeu complet le confirme à sa manière : `236544` géodésiques pour l'antipode des deux tables, soit 16 × 16 × 924, les deux familles de seize mots réduits entrelacées par les `C(12,6) = 924` ordres de pas.
3. Parce que l'univers est **fini et épuisé**, pas échantillonné. Le §5.2 compare une à une les `24 × 24 = 576` paires d'ordres stricts dans le permutaèdre et dans l'espace à rangs complet, et la sortie donne `Paires ou l'espace complet (murs autorises) est PLUS COURT que le permutaedre : 0`. La structure produit fait le reste : chaque générateur ne touche qu'un seul côté, donc `d_jeu(G,H) = d_cote(row_G,row_H) + d_cote(col_G,col_H)` dans **chaque** univers ; le verdict par côté couvre alors `576 × 576 = 331776` paires de chambres sans exception, comme l'imprime la même sortie. Le diamètre le confirme : l'univers complet est connexe sur `5625 sommets` et garde `Diametre de l'univers complet : 12`, identique au permutaèdre des chambres. Un 0 trouvé sur un échantillon n'aurait rien prouvé ; ici l'énumération épuise l'espace et la décomposition par produit transporte le résultat du côté au jeu entier — c'est une preuve par exhaustivité, pas l'absence de contre-exemple trouvé.

</details>

## Pour aller plus loin

- **Le substrat Life (GOL-01)** — là où la loi II a été franchie pour la première fois (la machine *redécouvre* le glider au lieu de le recopier).
- **Le troisième substrat, 3×2 (Metajeux-06)** — où les murs *raccourcissent* bien : le contraste avec le résultat négatif d'ici.
- **L'épaisseur des murs en forme extensive** (§6) : engagements observables (promesse, menace programmée) à la manière de Schelling.